# 01 · The bench

**The problem this project attacks.** Generative models are increasingly used
as priors inside scientific inference. A diffusion or flow model is trained on
simulations, then steered with data so that its samples follow a posterior
distribution over maps, images, or molecules. The steering methods in actual
use are approximations without guarantees, and on real data their output
cannot be checked, because checking would require the true posterior. That is
the thing being computed. So results ship with error bars nobody has verified,
and the diagnostics offered as verification have rarely been tested themselves.

**The move.** Build a world where the true posterior is known exactly, at the
scale where these methods actually operate (fields with thousands of
dimensions). Use it for two audits. First, measure how wrong each sampler
actually is, and why. Second, test the diagnostics themselves: force each one
to pass on a provably perfect sampler before it is allowed an opinion, then
show it constructed failures with exactly known damage and record what it
catches.

This notebook builds the bench and demonstrates its exactness. The next three
walk through the sampler anatomy (02), the trial of the certificates (03), and
the nonlinear extension with MCMC gold standards (04).

## The construction

The prior is a Gaussian random field, a structured random texture with a
cosmology-flavored power spectrum. Written in its Fourier basis it is a list
of independent Gaussians, one per spatial frequency. The data enter through a
quadratic reward. With observation y, smoothing operator A, and noise level s,
the steered target is

$$\sigma(x) \propto p(x)\, e^{r(x)/\beta}, \qquad r(x) = -\|Ax - y\|^2 / (2 s^2).$$

In words: take the prior probability of a map, multiply by how well that map
explains the data, with the steering strength set by beta. For this family the
tilted target is the classical Wiener posterior. Its mean, its variance, the
score, the optimal twist, and the exact distance of any Gaussian approximation
from it are all available in closed form, per Fourier mode, at any size. A
sampler's error can therefore be measured with error bars of zero.

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
RES = ROOT / "results"

def rows(name):
    p = RES / name
    if not p.exists():
        print(f"({name} not found, cell skipped)")
        return []
    return [json.loads(l) for l in p.open()]

from tilt_audit import tilt
from tilt_audit.fields import (grid_to_z, make_basis, make_pk,
                               smoothing_operator, unpack, sample_prior_z)
import jax
import jax.numpy as jnp

n = 64
basis = make_basis(n)
Pz = jnp.asarray(grid_to_z(make_pk(basis), basis))
az = jnp.asarray(smoothing_operator(basis))
y, z_truth = tilt.make_observation(jax.random.PRNGKey(999), Pz, az, 0.5)
b = float(tilt.calibrate_b(Pz, az, y, target_shift=1.0))
mu, Sig = tilt.posterior_params(Pz, az, y, b)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, field, title in [
        (axes[0], unpack(z_truth, basis), "a prior draw (the 'truth')"),
        (axes[1], unpack(y, basis), "the observation y (smoothed + noisy)"),
        (axes[2], unpack(mu, basis), "exact posterior mean (closed form)")]:
    im = ax.imshow(np.asarray(field), cmap="RdBu_r")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"dimensions: {int(Pz.shape[0])}, steering strength b = {b:.4g}")

## Exactness, demonstrated

The oracle sampler draws directly from the closed-form posterior. If the bench
is what we claim, the oracle's empirical moments must match the formulas to
within Monte Carlo noise, mode by mode. This same pattern, called a null gate
in this project, is applied to every instrument before it is trusted: a test
that does not pass on the exact answer is not allowed to judge anything.

In [ ]:
from tilt_audit import samplers

res = samplers.oracle(jax.random.PRNGKey(0), Pz, az, y, b, N=4096, T=1, tf=9.0)
z = np.asarray(res["z"])
zscores = (z.mean(0) - np.asarray(mu)) / np.sqrt(np.asarray(Sig) / z.shape[0])
vratio = z.var(0) / np.asarray(Sig)
print(f"per-mode mean z-scores: max |z| = {np.abs(zscores).max():.2f} "
      f"(expected < ~4.3 for {z.shape[1]} modes under pure noise)")
print(f"per-mode variance ratios: median = {np.median(vratio):.4f} "
      f"(expected 1.000 +- {np.sqrt(2/z.shape[0]):.3f})")

Both numbers sit at their theoretical noise floors. The bench is exact,
and every result in the next notebooks inherits that property: when a sampler
reads as 10 times worse than the floor, that is a measurement, not an estimate.

**Where the data live.** Every experiment appends rows to `results/*.jsonl`
with full configuration. Every figure in the repository regenerates from those
files. The prediction ledger (`RESEARCH_LOG.md`) holds the pre-registered
expectations for each experiment, frozen and pushed publicly before the runs,
and scored afterwards. Continue with notebook 02 for what the samplers
actually do.